In [1]:
#| default_exp core.layers
#| export

import numpy as np

# Import from TinyTorch package (previous modules must be completed and exported)
from tinytorch.core.tensor import Tensor
from tinytorch.core.activations import ReLU, Sigmoid

# Constants for weight initialization
# Note: True Xavier/Glorot uses sqrt(2/(fan_in+fan_out)), but we use the simpler
# LeCun-style sqrt(1/fan_in) for pedagogical clarity. Both achieve stable gradients.
INIT_SCALE_FACTOR = 1.0  # LeCun-style initialization: sqrt(1/fan_in)
HE_SCALE_FACTOR = 2.0  # He initialization uses sqrt(2/fan_in) for ReLU

# Constants for dropout
DROPOUT_MIN_PROB = 0.0  # Minimum dropout probability (no dropout)
DROPOUT_MAX_PROB = 1.0  # Maximum dropout probability (drop everything)

In [4]:
#| export
class Layer:
    """
    Base class for all neural network layers.

    All layers should inherit from this class and implement:
    - forward(x): Compute layer output
    - parameters(): Return list of trainable parameters

    The __call__ method is provided to make layers callable.
    """

    def forward(self, x):
        """
        Forward pass through the layer.

        Args:
            x: Input tensor

        Returns:
            Output tensor after transformation
        """
        raise NotImplementedError(
            f"forward() not implemented in {self.__class__.__name__}\n"
            f"  ❌ The Layer base class requires subclasses to implement forward()\n"
            f"  💡 forward() defines how input data is transformed by this layer\n"
            f"  🔧 Add this method to your class:\n"
            f"     def forward(self, x):\n"
            f"         # Your transformation logic here\n"
            f"         return transformed_x"
        )
    
    def __call__(self, x, *args, **kwargs):
        """Allow layer to be called like a function."""
        return self.forward(x, *args, **kwargs)
    
    def parameters(self):
        """
        Return list of trainable parameters.

        Returns:
            List of Tensor objects (weights and biases)
        """
        return []  # Base class has no parameters
    
    def __repr__(self):
        """String representation of the layer."""
        return f"{self.__class__.__name__}()"

In [5]:
def greet(*args, **kwargs):
    print(args)    # tuple
    print(kwargs)  # dict

greet("hello", "world", name="Alice", age=25)
# ("hello", "world")
# {"name": "Alice", "age": 25}

('hello', 'world')
{'name': 'Alice', 'age': 25}


In [6]:
#| export
class Linear(Layer):
    """
    Linear (fully connected) layer: y = xW + b

    This is the fundamental building block of neural networks.
    Applies a linear transformation to incoming data.
    """

    def __init__(self, in_features, out_features, bias=True):
        """
        Initialize linear layer with proper weight initialization.

        TODO: Initialize weights and bias with proper scaling

        APPROACH:
        1. Create weight matrix (in_features, out_features) with LeCun scaling
        2. Create bias vector (out_features,) initialized to zeros if bias=True
        3. Store as Tensor objects for use in forward pass

        EXAMPLE:
        >>> layer = Linear(784, 10)  # MNIST classifier final layer
        >>> print(layer.weight.shape)
        (784, 10)
        >>> print(layer.bias.shape)
        (10,)

        HINTS:
        - LeCun-style init: scale = sqrt(1/in_features)
        - Use np.random.randn() for normal distribution
        - bias=None when bias=False
        """

        self.in_features = in_features
        self.out_features = out_features

        # lecun-style init for stable gradients
        scale = np.sqrt(INIT_SCALE_FACTOR / in_features)
        weight_data = np.random.randn(in_features, out_features) * scale
        self.weight = Tensor(weight_data)

        # init bias to zeros or None
        if bias:
            bias_data = np.zeros(out_features)
            self.bias = Tensor(bias_data)
        else:
            self.bias = None

    def forward(self, x):
        """
        Forward pass through linear layer.

        TODO: Implement y = xW + b

        APPROACH:
        1. Matrix multiply input with weights: xW
        2. Add bias if it exists
        3. Return result as new Tensor

        EXAMPLE:
        >>> layer = Linear(3, 2)
        >>> x = Tensor([[1, 2, 3], [4, 5, 6]])  # 2 samples, 3 features
        >>> y = layer.forward(x)
        >>> print(y.shape)
        (2, 2)  # 2 samples, 2 outputs

        HINTS:
        - Use tensor.matmul() for matrix multiplication
        - Handle bias=None case
        - Broadcasting automatically handles bias addition
        """

        # linear transformation: y = xW
        output = x.matmul(self.weight)

        # add bias if present
        if self.bias is not None:
            output = output + self.bias

        return output
    
    def parameters(self):
        """
        Return list of trainable parameters.

        TODO: Return all tensors that need gradients

        APPROACH:
        1. Start with weight (always present)
        2. Add bias if it exists
        3. Return as list for optimizer

        EXAMPLE:
        >>> layer = Linear(10, 5)
        >>> params = layer.parameters()
        >>> len(params)
        2  # [weight, bias]
        >>> layer_no_bias = Linear(10, 5, bias=False)
        >>> len(layer_no_bias.parameters())
        1  # [weight only]

        HINTS:
        - Create list starting with self.weight
        - Check if self.bias is not None before appending
        - Return the complete list
        """

        params = [self.weight]
        if self.bias is not None:
            params.append(self.bias)
        return params
    
    def __repr__(self):
        """String representation for debugging."""
        bias_str = f", bias={self.bias is not None}"
        return f"Linear(in_features={self.in_features}, out_features={self.out_features}{bias_str})"
        

In [10]:
def test_unit_linear_layer():
    """🧪 Test Linear layer implementation."""
    print("🧪 Unit Test: Linear Layer...")

    # Test layer creation
    layer = Linear(784, 256)
    print(layer)
    assert layer.in_features == 784
    assert layer.out_features == 256
    assert layer.weight.shape == (784, 256)
    assert layer.bias.shape == (256,)

    # Test LeCun-style initialization (weights should be reasonably scaled)
    weight_std = np.std(layer.weight.data)
    expected_std = np.sqrt(INIT_SCALE_FACTOR / 784)
    assert 0.5 * expected_std < weight_std < 2.0 * expected_std, f"Weight std {weight_std} not close to expected {expected_std}"

    # Test bias initialization (should be zeros)
    assert np.allclose(layer.bias.data, 0), "Bias should be initialized to zeros"

    # Test forward pass
    x = Tensor(np.random.randn(32, 784))  # Batch of 32 samples
    y = layer.forward(x)
    assert y.shape == (32, 256), f"Expected shape (32, 256), got {y.shape}"

    # Test no bias option
    layer_no_bias = Linear(10, 5, bias=False)
    assert layer_no_bias.bias is None
    params = layer_no_bias.parameters()
    assert len(params) == 1  # Only weight, no bias

    # Test parameters method
    params = layer.parameters()
    assert len(params) == 2  # Weight and bias
    assert params[0] is layer.weight
    assert params[1] is layer.bias

    print("✅ Linear layer works correctly!")

if __name__ == "__main__":
    test_unit_linear_layer()

🧪 Unit Test: Linear Layer...
Linear(in_features=784, out_features=256, bias=True)
✅ Linear layer works correctly!


In [11]:
def test_unit_edge_cases_linear():
    """🧪 Test Linear layer edge cases."""
    print("🧪 Edge Case Tests: Linear Layer...")

    layer = Linear(10, 5)

    # Test single sample (should handle 2D input)
    x_2d = Tensor(np.random.randn(1, 10))
    y = layer.forward(x_2d)
    assert y.shape == (1, 5), "Should handle single sample"

    # Test zero batch size (edge case)
    x_empty = Tensor(np.random.randn(0, 10))
    y_empty = layer.forward(x_empty)
    assert y_empty.shape == (0, 5), "Should handle empty batch"

    # Test numerical stability with large weights
    layer_large = Linear(10, 5)
    layer_large.weight.data = np.ones((10, 5)) * 100  # Large but not extreme
    x = Tensor(np.ones((1, 10)))
    y = layer_large.forward(x)
    assert not np.any(np.isnan(y.data)), "Should not produce NaN with large weights"
    assert not np.any(np.isinf(y.data)), "Should not produce Inf with large weights"

    # Test with no bias
    layer_no_bias = Linear(10, 5, bias=False)
    x = Tensor(np.random.randn(4, 10))
    y = layer_no_bias.forward(x)
    assert y.shape == (4, 5), "Should work without bias"

    print("✅ Edge cases handled correctly!")

if __name__ == "__main__":
    test_unit_edge_cases_linear()

🧪 Edge Case Tests: Linear Layer...
✅ Edge cases handled correctly!


In [12]:
def test_unit_parameter_collection_linear():
    """🧪 Test Linear layer parameter collection."""
    print("🧪 Parameter Collection Test: Linear Layer...")

    layer = Linear(10, 5)

    # Verify parameter collection works
    params = layer.parameters()
    assert len(params) == 2, "Should return 2 parameters (weight and bias)"
    assert params[0].shape == (10, 5), "First param should be weight"
    assert params[1].shape == (5,), "Second param should be bias"

    # Test layer without bias
    layer_no_bias = Linear(10, 5, bias=False)
    params_no_bias = layer_no_bias.parameters()
    assert len(params_no_bias) == 1, "Should return 1 parameter (weight only)"

    print("✅ Parameter collection works correctly!")

if __name__ == "__main__":
    test_unit_parameter_collection_linear()

🧪 Parameter Collection Test: Linear Layer...
✅ Parameter collection works correctly!


In [13]:
#| export
class Dropout(Layer):
    """
    Dropout layer for regularization.

    During training: randomly zeros elements with probability p, scales survivors by 1/(1-p)
    During inference: passes input through unchanged

    This prevents overfitting by forcing the network to not rely on specific neurons.
    """
    def __init__(self, p=0.5):
        """
        Initialize dropout layer.

        TODO: Store dropout probability and validate range

        APPROACH:
        1. Validate p is between 0.0 and 1.0 (inclusive)
        2. Raise ValueError if out of range
        3. Store p as instance attribute

        Args:
            p: Probability of zeroing each element (0.0 = no dropout, 1.0 = zero everything)

        EXAMPLE:
        >>> dropout = Dropout(0.5)  # Zero 50% of elements during training
        >>> dropout.p
        0.5

        HINTS:
        - Use DROPOUT_MIN_PROB and DROPOUT_MAX_PROB constants for validation
        - Check: DROPOUT_MIN_PROB <= p <= DROPOUT_MAX_PROB
        - Raise descriptive ValueError if invalid
        """
        if not DROPOUT_MIN_PROB <= p <= DROPOUT_MAX_PROB:
            raise ValueError(
                f"Invalid dropout probability: {p}\n"
                f"  ❌ p must be between {DROPOUT_MIN_PROB} and {DROPOUT_MAX_PROB}\n"
                f"  💡 p is the probability of DROPPING a neuron (not keeping it!)\n"
                f"     p=0.0 means keep all neurons (no dropout)\n"
                f"     p=0.5 means drop 50% of neurons randomly\n"
                f"     p=1.0 means drop all neurons (zero output)\n"
                f"  🔧 Common values: Dropout(0.1) for light, Dropout(0.3) for moderate, Dropout(0.5) for aggressive"
            )
        self.p = p

    def _should_apply_dropout(self, training):
        """
        Determine whether dropout should be applied.

        Dropout is a training-time technique. During inference the full
        network is used, so dropout is skipped. It is also skipped when p=0
        (no neurons are dropped) since the result would be the identity.

        TODO: Return True only when dropout should actually modify the input

        APPROACH:
        1. Check if we are in training mode
        2. Check if dropout probability is greater than zero

        EXAMPLE:
        >>> d = Dropout(0.5)
        >>> d._should_apply_dropout(training=True)
        True
        >>> d._should_apply_dropout(training=False)
        False

        HINT: Both conditions must be true for dropout to apply
        """
        return training and self.p > DROPOUT_MIN_PROB

    def _generate_dropout_mask(self, shape):
        """
        Generate a random dropout mask with inverted scaling.

        The mask has the same shape as the input. Each element is either
        0 (dropped) or 1/(1-p) (kept and scaled). Scaling at training time
        keeps the expected value of each element unchanged, so no adjustment
        is needed at inference. This trick is called "inverted dropout."

        ```
        Example with p=0.5 (keep_prob=0.5, scale=2.0):
        random draw:  [0.3,  0.8,  0.1,  0.6]
                        ↓     ↓     ↓     ↓
        keep?         [yes,  no,  yes,  no ]   (< 0.5?)
                        ↓     ↓     ↓     ↓
        mask:         [2.0,  0.0,  2.0,  0.0]  (kept × scale, dropped × 0)
        ```

        TODO: Build the scaled binary mask

        APPROACH:
        1. Compute keep_prob = 1 - p
        2. Draw uniform random values and threshold at keep_prob
        3. Convert the boolean mask to float and scale by 1/keep_prob

        EXAMPLE:
        >>> d = Dropout(0.5)
        >>> mask = d._generate_dropout_mask((4,))
        >>> mask.shape
        (4,)

        HINTS:
        - np.random.random(shape) gives uniform [0, 1) values
        - Threshold with < keep_prob to get a boolean mask
        - Scale factor is 1.0 / keep_prob
        """

        keep_prob = 1.0 - self.p
        binary_mask = (np.random.random(shape) < keep_prob).astype(np.float32)
        scale = 1.0 / keep_prob
        return Tensor(binary_mask * scale) # no dropout neuron are amplified, dropout neuron become 0
    
    def forward(self, x, training=True):
        """
        Forward pass through dropout layer.

        Composes the two helpers: first decide whether dropout applies,
        then generate and apply the mask if it does.

        TODO: Implement dropout forward pass

        APPROACH:
        1. Use _should_apply_dropout to check if dropout is needed
        2. Handle the special case p=1 (drop everything)
        3. Use _generate_dropout_mask to create the scaled mask
        4. Element-wise multiply input by the mask

        EXAMPLE:
        >>> dropout = Dropout(0.5)
        >>> x = Tensor([1, 2, 3, 4])
        >>> y_train = dropout.forward(x, training=True)   # Some elements zeroed
        >>> y_eval = dropout.forward(x, training=False)   # All elements preserved

        HINTS:
        - _should_apply_dropout returns False for inference or p=0
        - When p=1.0 every element is dropped (return zeros)
        - Multiply x by the mask tensor for the final output
        """

        if not self._should_apply_dropout(training):
            return x
        
        if self.p == DROPOUT_MAX_PROB:
            return Tensor(np.zeros_like(x.data))
        
        mask = self._generate_dropout_mask(x.data.shape)
        return x * mask
    
    def __call__(self, x, training=True):
        """Allows the layer to be called like a function."""
        return self.forward(x, training)
    
    def parameters(self):
        """Dropout has no parameters."""
        return []
    
    def __repr__(self):
        return f"Dropout(p={self.p})"
    
    



In [14]:
def test_unit_should_apply_dropout():
    """🧪 Test _should_apply_dropout decision logic."""
    print("🧪 Unit Test: Dropout Decision Logic...")

    # Standard dropout (p=0.5) in training mode should apply
    d = Dropout(0.5)
    assert d._should_apply_dropout(training=True) is True, \
        "Dropout(0.5) should apply during training"

    # Same dropout in inference mode should NOT apply
    assert d._should_apply_dropout(training=False) is False, \
        "Dropout should not apply during inference"

    # Zero dropout (p=0) should never apply, even in training
    d_zero = Dropout(0.0)
    assert d_zero._should_apply_dropout(training=True) is False, \
        "Dropout(0.0) should never apply (no neurons to drop)"

    # Full dropout (p=1.0) in training mode should apply
    d_full = Dropout(1.0)
    assert d_full._should_apply_dropout(training=True) is True, \
        "Dropout(1.0) should apply during training"

    # Full dropout in inference mode should NOT apply
    assert d_full._should_apply_dropout(training=False) is False, \
        "Even Dropout(1.0) should not apply during inference"

    print("✅ Dropout decision logic works correctly!")

if __name__ == "__main__":
    test_unit_should_apply_dropout()

🧪 Unit Test: Dropout Decision Logic...
✅ Dropout decision logic works correctly!


In [15]:
def test_unit_generate_dropout_mask():
    """🧪 Test _generate_dropout_mask output properties."""
    print("🧪 Unit Test: Dropout Mask Generation...")

    d = Dropout(0.5)
    np.random.seed(42)
    mask = d._generate_dropout_mask((1000,))

    # Shape must match the requested shape
    assert mask.shape == (1000,), f"Expected shape (1000,), got {mask.shape}"

    # Every element must be either 0.0 or 2.0 (= 1/(1-0.5))
    unique_vals = set(np.unique(mask.data))
    assert unique_vals <= {0.0, 2.0}, \
        f"Mask values should be {{0.0, 2.0}}, got {unique_vals}"

    # Statistically, about 50% should survive (3-sigma tolerance)
    non_zero = np.count_nonzero(mask.data)
    std_err = np.sqrt(1000 * 0.5 * 0.5)
    assert 500 - 3 * std_err < non_zero < 500 + 3 * std_err, \
        f"Expected ~500 survivors, got {non_zero}"

    # Test with different dropout probability
    d2 = Dropout(0.3)
    np.random.seed(123)
    mask2 = d2._generate_dropout_mask((2000,))

    # Values should be 0.0 or 1/(1-0.3) ≈ 1.4286
    expected_scale = 1.0 / 0.7
    non_zero_vals = mask2.data[mask2.data != 0.0]
    assert np.allclose(non_zero_vals, expected_scale), \
        f"Surviving values should be {expected_scale:.4f}, got {np.unique(non_zero_vals)}"

    # About 70% should survive for p=0.3
    survival_rate = np.count_nonzero(mask2.data) / 2000
    assert 0.60 < survival_rate < 0.80, \
        f"Expected ~70% survival for p=0.3, got {survival_rate:.1%}"

    print("✅ Dropout mask generation works correctly!")

if __name__ == "__main__":
    test_unit_generate_dropout_mask()

🧪 Unit Test: Dropout Mask Generation...
✅ Dropout mask generation works correctly!
